# Generador de Dataset Limpio
## Proyecto: Radar Digital - Sistema de Análisis de Hábitos Digitales

### Este notebook realiza el procesamiento inicial del dataset:
 - Carga y análisis exploratorio
 - Limpieza de datos (sin perder la clase "None")
 - Selección de variables relevantes
 - Generación de variables derivadas
 - Conservación de copias sin escalar para el score de riesgo compuesto
 - Guardado del dataset limpio y artefactos necesarios


## 1. Importación de Librerías

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy import stats
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 2. Configuración de Rutas

In [3]:
ROOT_DIR = os.path.abspath('..')
DATASET_PATH = os.path.join(ROOT_DIR, 'datasets', 'Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv')
PROCESADO_DIR = os.path.join(ROOT_DIR, 'datasets', 'limpio_procesado')
ARTIFACTS_DIR = os.path.join(ROOT_DIR, 'datasets', 'artifacts')

os.makedirs(PROCESADO_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"📁 Dataset original: {DATASET_PATH}")
print(f"📂 Dataset procesado: {PROCESADO_DIR}")
print(f"📂 Artefactos: {ARTIFACTS_DIR}")

📁 Dataset original: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv
📂 Dataset procesado: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/limpio_procesado
📂 Artefactos: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/artifacts


## 3. Carga del Dataset

**Importante:** se usa `keep_default_na=False` para que pandas no interprete el texto `"None"` como valor nulo.

In [4]:
df_raw = pd.read_csv(DATASET_PATH, keep_default_na=False)

print(f"📊 Dimensiones del dataset: {df_raw.shape}")
print(f"📋 Columnas disponibles: {df_raw.columns.tolist()}")
df_raw.head()

📊 Dimensiones del dataset: (7500, 16)
📋 Columnas disponibles: ['transaction_id', 'user_id', 'age', 'gender', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'stress_level', 'academic_work_impact', 'addiction_level', 'addicted_label']


,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,TXN00001,U00001,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,None,0
1,TXN00002,U00002,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,None,0
2,TXN00003,U00003,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,Mild,0
3,TXN00004,U00004,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,Moderate,1
4,TXN00005,U00005,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,Severe,1


In [5]:
print("📊 Información del dataset:")
df_raw.info()

📊 Información del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   transaction_id           7500 non-null   object 
 1   user_id                  7500 non-null   object 
 2   age                      7500 non-null   int64  
 3   gender                   7500 non-null   object 
 4   daily_screen_time_hours  7500 non-null   float64
 5   social_media_hours       7500 non-null   float64
 6   gaming_hours             7500 non-null   float64
 7   work_study_hours         7500 non-null   float64
 8   sleep_hours              7500 non-null   float64
 9   notifications_per_day    7500 non-null   int64  
 10  app_opens_per_day        7500 non-null   int64  
 11  weekend_screen_time      7500 non-null   float64
 12  stress_level             7500 non-null   object 
 13  academic_work_impact     7500 non-null   object 
 1

In [6]:
print("📊 Estadísticas descriptivas:")
display(df_raw.describe())

📊 Estadísticas descriptivas:


,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,addicted_label
count,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000
mean,26.568800,7.499912,3.273484,2.014183,3.242420,6.737561,134.257333,97.832400,9.243827,0.707733
std,5.197108,2.609188,1.585342,1.146039,1.600765,1.283605,66.586883,48.423349,2.718281,0.454835
min,18.000000,3.000000,0.500000,0.000000,0.500000,4.500000,20.000000,15.000000,3.580000,0.000000
25%,22.000000,5.220000,1.910000,1.020000,1.850000,5.630000,76.000000,55.000000,6.960000,0.000000
50%,27.000000,7.525000,3.270000,2.040000,3.230000,6.720000,134.000000,98.000000,9.260000,1.000000
75%,31.000000,9.810000,4.630000,2.990000,4.640000,7.840000,191.000000,140.000000,11.540000,1.000000
max,35.000000,12.000000,6.000000,4.000000,6.000000,9.000000,250.000000,180.000000,14.880000,1.000000


## 4. Selección de Variables

**Nota:** `weekend_screen_time` se mantiene porque es necesaria para calcular `screen_time_weekend_difference`. Luego se eliminará si es redundante.

In [7]:
VARIABLES_AUTOMATICAS = [
    'daily_screen_time_hours',
    'social_media_hours',
    'gaming_hours',
    'work_study_hours',
    'sleep_hours',
    'notifications_per_day',
    'app_opens_per_day',
    'weekend_screen_time'  # Necesaria para calcular diferencia
]

TARGET_VARIABLE = 'addiction_level'

VARIABLES_DESCARTADAS = [
    'transaction_id',
    'user_id',
    'stress_level',
    'academic_work_impact',
    'addicted_label'
]

print("✅ Variables seleccionadas:")
for var in VARIABLES_AUTOMATICAS:
    print(f"   - {var}")
print(f"\n🎯 Variable objetivo: {TARGET_VARIABLE}")
print(f"\n❌ Variables descartadas: {VARIABLES_DESCARTADAS}")

✅ Variables seleccionadas:
   - daily_screen_time_hours
   - social_media_hours
   - gaming_hours
   - work_study_hours
   - sleep_hours
   - notifications_per_day
   - app_opens_per_day
   - weekend_screen_time

🎯 Variable objetivo: addiction_level

❌ Variables descartadas: ['transaction_id', 'user_id', 'stress_level', 'academic_work_impact', 'addicted_label']


## 5. Limpieza de Registros

In [8]:
df_temp = df_raw[VARIABLES_AUTOMATICAS + [TARGET_VARIABLE]].copy()

print("="*60)
print("🧹 LIMPIEZA DE REGISTROS")
print("="*60)
print(f"📊 Registros totales: {len(df_temp)}")

# Verificar valores vacíos
vacios = (df_temp == '').sum()
vacios = vacios[vacios > 0]
if len(vacios) > 0:
    print(f"\n🔍 Celdas vacías encontradas: {vacios.to_dict()}")
    df_temp = df_temp.replace('', np.nan)

# Eliminar NaN y duplicados
df_clean = df_temp.dropna()
duplicados = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

print(f"\n✅ Registros después de limpiar: {len(df_clean)}")
if duplicados > 0:
    print(f"   (Eliminados {len(df_temp) - len(df_clean)} registros, {duplicados} duplicados)")

# Distribución de clases
print("\n📊 Distribución de addiction_level:")
dist = df_clean[TARGET_VARIABLE].value_counts()
for level, count in dist.items():
    print(f"   {level}: {count} ({count/len(df_clean)*100:.1f}%)")

df_selected = df_clean.copy()
print("="*60)

🧹 LIMPIEZA DE REGISTROS
📊 Registros totales: 7500

✅ Registros después de limpiar: 7500

📊 Distribución de addiction_level:
   Moderate: 2874 (38.3%)
   Severe: 2434 (32.5%)
   Mild: 1373 (18.3%)
   None: 819 (10.9%)


In [9]:
print(f"📊 Dimensiones del dataset limpio: {df_selected.shape}")
df_selected.head()

📊 Dimensiones del dataset limpio: (7500, 9)


,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,addiction_level
0,3.23,2.01,0.89,4.55,7.55,248,154,3.95,None
1,5.09,3.81,2.24,4.44,7.66,127,71,6.71,None
2,6.06,1.36,3.83,2.35,4.92,44,106,8.68,Mild
3,7.83,5.85,1.51,3.54,8.23,178,107,9.77,Moderate
4,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Severe


## 6. Generación de Variables Derivadas

In [10]:
# 1. screen_time_weekend_difference: diferencia fin de semana - semana
df_selected['screen_time_weekend_difference'] = df_selected['weekend_screen_time'] - df_selected['daily_screen_time_hours']

# 2-4. Ratios
df_selected['social_media_ratio'] = df_selected['social_media_hours'] / (df_selected['daily_screen_time_hours'] + 0.01)
df_selected['gaming_ratio'] = df_selected['gaming_hours'] / (df_selected['daily_screen_time_hours'] + 0.01)
df_selected['work_study_ratio'] = df_selected['work_study_hours'] / (df_selected['daily_screen_time_hours'] + 0.01)

# 5. usage_intensity_index
df_selected['usage_intensity_index'] = (df_selected['notifications_per_day'] + df_selected['app_opens_per_day']) / 100

# 6. productivity_ratio: productivo vs entretenimiento
df_selected['productivity_ratio'] = df_selected['work_study_hours'] / (
    df_selected['social_media_hours'] + df_selected['gaming_hours'] + 0.01
)

# 7. notification_intensity
df_selected['notification_intensity'] = df_selected['notifications_per_day'] / (
    df_selected['daily_screen_time_hours'] + 0.01
)

# 8. total_usage_composite: índice compuesto de uso
df_selected['total_usage_composite'] = (
    df_selected['daily_screen_time_hours'] * 0.3 +
    df_selected['social_media_hours'] * 0.25 +
    df_selected['gaming_hours'] * 0.15 +
    df_selected['work_study_hours'] * 0.1 +
    (df_selected['notifications_per_day'] / 100) * 0.2
)

# 9. Desbalance sueño-pantalla
df_selected['sleep_screen_imbalance'] = df_selected['daily_screen_time_hours'] / (df_selected['sleep_hours'] + 0.01)

# 10. Dominancia de entretenimiento
df_selected['entertainment_dominance'] = (df_selected['social_media_hours'] + df_selected['gaming_hours']) / (df_selected['work_study_hours'] + 0.01)

# 11. Compulsividad digital
df_selected['digital_compulsivity'] = df_selected['app_opens_per_day'] / (df_selected['daily_screen_time_hours'] + 0.01)

DERIVED_VARS = [
    'screen_time_weekend_difference', 'social_media_ratio',
    'gaming_ratio', 'work_study_ratio', 'usage_intensity_index', 
    'productivity_ratio', 'notification_intensity', 'total_usage_composite',
    'sleep_screen_imbalance', 'entertainment_dominance', 'digital_compulsivity'
]

print(f"✅ {len(DERIVED_VARS)} variables derivadas creadas")

✅ 11 variables derivadas creadas


In [11]:
print("📊 Estadísticas de variables derivadas:")
display(df_selected[DERIVED_VARS].describe())

📊 Estadísticas de variables derivadas:


,screen_time_weekend_difference,social_media_ratio,gaming_ratio,work_study_ratio,usage_intensity_index,productivity_ratio,notification_intensity,total_usage_composite,sleep_screen_imbalance,entertainment_dominance,digital_compulsivity
count,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000
mean,1.743915,0.503831,0.309998,0.498132,2.320897,0.750480,20.673157,3.963229,1.152929,2.386556,15.000501
std,0.719982,0.338426,0.229769,0.334505,0.825123,0.646788,14.067393,0.921525,0.464040,2.183236,10.116595
min,0.500000,0.042772,0.000000,0.045038,0.370000,0.053135,1.700680,1.235000,0.338889,0.095477,1.256281
25%,1.130000,0.254008,0.135462,0.245215,1.730000,0.348183,10.262528,3.255000,0.778224,1.071054,7.523742
50%,1.730000,0.437225,0.270692,0.430947,2.330000,0.605223,17.936031,3.965000,1.115055,1.645193,13.017231
75%,2.372500,0.659329,0.418992,0.656224,2.920000,0.929506,27.068069,4.668125,1.460746,2.853598,19.612758
max,3.000000,1.953795,1.270968,1.928339,4.270000,10.275862,81.639344,6.390500,2.654867,18.431373,58.823529


## 7. Conservar copia sin escalar para el score de riesgo compuesto

`predictor.py` necesita valores en unidades reales (horas, notificaciones, etc.) para calcular el riesgo compuesto.

**Nota:** Solo se guardan `_raw` de las columnas que predictor.py necesita realmente.

In [12]:
COLUMNAS_PARA_RIESGO_COMPUESTO = [
    'daily_screen_time_hours',
    'sleep_hours',
    'social_media_hours',
    'notifications_per_day',
    'screen_time_weekend_difference',
]

for col in COLUMNAS_PARA_RIESGO_COMPUESTO:
    df_selected[f'{col}_raw'] = df_selected[col]

print("✅ Copias sin escalar guardadas (_raw)")
print(f"   Columnas: {[f'{col}_raw' for col in COLUMNAS_PARA_RIESGO_COMPUESTO]}")

# Verificar consistencia
for col in COLUMNAS_PARA_RIESGO_COMPUESTO:
    raw_col = f'{col}_raw'
    if (df_selected[col] == df_selected[raw_col]).all():
        print(f"   ✅ {raw_col} = {col} (sin escalar)")

✅ Copias sin escalar guardadas (_raw)
   Columnas: ['daily_screen_time_hours_raw', 'sleep_hours_raw', 'social_media_hours_raw', 'notifications_per_day_raw', 'screen_time_weekend_difference_raw']
   ✅ daily_screen_time_hours_raw = daily_screen_time_hours (sin escalar)
   ✅ sleep_hours_raw = sleep_hours (sin escalar)
   ✅ social_media_hours_raw = social_media_hours (sin escalar)
   ✅ notifications_per_day_raw = notifications_per_day (sin escalar)
   ✅ screen_time_weekend_difference_raw = screen_time_weekend_difference (sin escalar)


## 8. Análisis de la Variable Objetivo

In [13]:
print("📊 Distribución de addiction_level:")
target_counts = df_selected['addiction_level'].value_counts()
target_percentages = (target_counts / len(df_selected)) * 100

target_dist = pd.DataFrame({
    'Count': target_counts,
    'Percentage': target_percentages
})
display(target_dist)

# Test de balance de clases
class_counts = df_selected['addiction_level'].value_counts()
expected = len(df_selected) / len(class_counts)
chi2, p_value = stats.chisquare(class_counts)

print(f"\n📊 Balance de clases:")
print(f"   Chi-cuadrado: {chi2:.2f}")
print(f"   p-valor: {p_value:.4f}")
if p_value > 0.05:
    print("   ✅ Clases balanceadas")
else:
    print("   ⚠️  Clases NO balanceadas (considerar balanceo en el Clasificador)")

plt.figure(figsize=(10, 6))
sns.countplot(data=df_selected, x='addiction_level', order=['None', 'Mild', 'Moderate', 'Severe'])
plt.title('Distribución de Niveles de Adicción')
plt.xlabel('Nivel de Adicción')
plt.ylabel('Cantidad de Usuarios')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'class_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()

📊 Distribución de addiction_level:


,Count,Percentage
addiction_level,,
Moderate,2874,38.320000
Severe,2434,32.453333
Mild,1373,18.306667
None,819,10.920000



📊 Balance de clases:
   Chi-cuadrado: 1428.07
   p-valor: 0.0000
   ⚠️  Clases NO balanceadas (considerar balanceo en el Clasificador)


## 9. Codificación de Variable Categórica

**Importante:** Se fuerza el orden de severidad `None < Mild < Moderate < Severe` para que el índice numérico corresponda a la severidad real.

In [14]:
ORDEN_SEVERIDAD = ['None', 'Mild', 'Moderate', 'Severe']

label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(ORDEN_SEVERIDAD)

df_selected['addiction_level_encoded'] = label_encoder.transform(df_selected['addiction_level'])

class_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("📊 Mapeo de clases (orden de severidad):")
for class_name, class_id in class_mapping.items():
    print(f"   {class_name} → {class_id}")

df_selected[['addiction_level', 'addiction_level_encoded']].head()

📊 Mapeo de clases (orden de severidad):
   None → 0
   Mild → 1
   Moderate → 2
   Severe → 3


,addiction_level,addiction_level_encoded
0,None,0
1,None,0
2,Mild,1
3,Moderate,2
4,Severe,3


## 10. Normalización de Variables Numéricas

In [15]:
COLUMNAS_RAW = [f'{col}_raw' for col in COLUMNAS_PARA_RIESGO_COMPUESTO]
FEATURE_COLUMNS = [
    col for col in df_selected.columns
    if col not in ['addiction_level', 'addiction_level_encoded'] + COLUMNAS_RAW
]

print(f"📊 Variables a normalizar: {len(FEATURE_COLUMNS)}")

📊 Variables a normalizar: 19


In [16]:
scaler = StandardScaler()
df_scaled = df_selected.copy()
df_scaled[FEATURE_COLUMNS] = scaler.fit_transform(df_selected[FEATURE_COLUMNS])

print("✅ Datos normalizados correctamente")

# Verificar normalización
print("\n📊 Estadísticas después de normalizar:")
stats_df = pd.DataFrame({
    'Media': df_scaled[FEATURE_COLUMNS].mean().round(6),
    'Desviación': df_scaled[FEATURE_COLUMNS].std().round(6)
}).T
display(stats_df)

✅ Datos normalizados correctamente

📊 Estadísticas después de normalizar:


,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,screen_time_weekend_difference,social_media_ratio,gaming_ratio,work_study_ratio,usage_intensity_index,productivity_ratio,notification_intensity,total_usage_composite,sleep_screen_imbalance,entertainment_dominance,digital_compulsivity
Media,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,-0.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,-0.000000
Desviación,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067,1.000067


## 11. Análisis de Correlación y Eliminación de Features Redundantes

In [17]:
correlations = df_scaled[FEATURE_COLUMNS + ['addiction_level_encoded']].corr()
target_corr = correlations['addiction_level_encoded'].drop('addiction_level_encoded').sort_values(ascending=False)

print("📊 Top 10 features con mayor correlación con el nivel de adicción:")
for feature, corr in target_corr.head(10).items():
    print(f"   {feature:30s} {corr:.4f}")

# Detectar features redundantes
corr_matrix = df_scaled[FEATURE_COLUMNS].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col1, col2, corr_matrix.loc[col1, col2])
             for col1 in upper.columns for col2 in upper.index
             if upper.loc[col2, col1] > 0.85]

print("\n🔍 Features altamente correlacionadas (>0.85):")
if high_corr:
    for col1, col2, corr_val in high_corr:
        print(f"   ⚠️ {col1} ↔ {col2}: {corr_val:.3f}")
    
    # Features a eliminar (redundantes)
    FEATURES_TO_DROP = ['weekend_screen_time']  # Eliminar porque es redundante con daily_screen_time_hours
    
    print(f"\n🔧 Eliminando features redundantes: {FEATURES_TO_DROP}")
    
    # Eliminar de FEATURE_COLUMNS
    FEATURE_COLUMNS_FILTERED = [f for f in FEATURE_COLUMNS if f not in FEATURES_TO_DROP]
    
    # Actualizar df_scaled
    df_scaled = df_scaled[FEATURE_COLUMNS_FILTERED + ['addiction_level', 'addiction_level_encoded'] + COLUMNAS_RAW]
    
    # Actualizar FEATURE_COLUMNS
    FEATURE_COLUMNS = FEATURE_COLUMNS_FILTERED
    
    print(f"✅ Features finales: {len(FEATURE_COLUMNS)} (reducción de {len(FEATURES_TO_DROP)})")
else:
    print("   ✅ No hay features redundantes")

# Guardar estadísticas del dataset
dataset_stats = {
    'total_rows': len(df_scaled),
    'features': len(FEATURE_COLUMNS),
    'class_distribution': df_scaled['addiction_level'].value_counts().to_dict(),
    'class_percentages': (df_scaled['addiction_level'].value_counts() / len(df_scaled) * 100).to_dict(),
    'correlations_with_target': target_corr.to_dict()
}

stats_path = os.path.join(ARTIFACTS_DIR, 'dataset_statistics.json')
with open(stats_path, 'w') as f:
    json.dump(dataset_stats, f, indent=4)
print(f"\n✅ Estadísticas guardadas en: {stats_path}")

# Matriz de correlación final
plt.figure(figsize=(14, 10))
sns.heatmap(df_scaled[FEATURE_COLUMNS + ['addiction_level_encoded']].corr(),
            cmap='coolwarm', annot=False, fmt='.2f',
            cbar_kws={'label': 'Correlación'})
plt.title('Matriz de Correlación con Nivel de Adicción', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'correlation_matrix.png'), dpi=300, bbox_inches='tight')
plt.close()
print("✅ Matriz de correlación guardada")

📊 Top 10 features con mayor correlación con el nivel de adicción:
   total_usage_composite          0.6253
   daily_screen_time_hours        0.5414
   weekend_screen_time            0.5234
   sleep_screen_imbalance         0.4559
   social_media_hours             0.3860
   entertainment_dominance        0.1221
   social_media_ratio             0.0624
   sleep_hours                    0.0327
   screen_time_weekend_difference 0.0139
   app_opens_per_day              0.0110

🔍 Features altamente correlacionadas (>0.85):
   ⚠️ weekend_screen_time ↔ daily_screen_time_hours: 0.964
   ⚠️ total_usage_composite ↔ daily_screen_time_hours: 0.855
   ⚠️ sleep_screen_imbalance ↔ daily_screen_time_hours: 0.853

🔧 Eliminando features redundantes: ['weekend_screen_time']
✅ Features finales: 18 (reducción de 1)

✅ Estadísticas guardadas en: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/artifacts/dataset_statistics.json
✅ Matriz de correlación guardada


## 12. Visualizaciones Adicionales

In [18]:
key_features = ['daily_screen_time_hours', 'sleep_hours', 'social_media_hours', 'notifications_per_day']

# Distribuciones por clase
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for idx, feature in enumerate(key_features):
    ax = axes[idx // 2, idx % 2]
    for level in ['None', 'Mild', 'Moderate', 'Severe']:
        data = df_selected[df_selected['addiction_level'] == level][feature]
        ax.hist(data, bins=20, alpha=0.5, label=level, density=True)
    ax.set_title(f'Distribución de {feature} por Nivel de Adicción')
    ax.set_xlabel(feature)
    ax.set_ylabel('Densidad')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'feature_distributions_by_class.png'), dpi=300, bbox_inches='tight')
plt.close()

# Boxplots por clase
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for idx, feature in enumerate(key_features):
    ax = axes[idx // 2, idx % 2]
    df_selected.boxplot(column=feature, by='addiction_level', ax=ax)
    ax.set_title(f'Boxplot de {feature} por Nivel de Adicción')
    ax.set_xlabel('Nivel de Adicción')
    ax.set_ylabel(feature)
plt.suptitle('')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'boxplots_by_class.png'), dpi=300, bbox_inches='tight')
plt.close()
print("✅ Visualizaciones guardadas")

✅ Visualizaciones guardadas


## 13. Verificación de Calidad de Datos

In [19]:
# Verificar rangos válidos
range_checks = {
    'daily_screen_time_hours': (0, 24),
    'sleep_hours': (0, 24),
    'social_media_hours': (0, 24),
    'gaming_hours': (0, 24),
    'work_study_hours': (0, 24),
    'notifications_per_day': (0, 1000),
    'app_opens_per_day': (0, 500),
}

print("\n🔍 Verificando rangos de valores:")
for col, (min_val, max_val) in range_checks.items():
    if col in df_selected.columns:
        col_min, col_max = df_selected[col].min(), df_selected[col].max()
        ok = col_min >= min_val and col_max <= max_val
        status = "✅" if ok else "⚠️"
        print(f"   {status} {col}: [{col_min:.2f}, {col_max:.2f}]")

# Detectar outliers
print("\n🔍 Outliers detectados (>3 desviaciones):")
outlier_count = 0
for col in FEATURE_COLUMNS:
    mean, std = df_selected[col].mean(), df_selected[col].std()
    outliers = df_selected[(df_selected[col] > mean + 3*std) | (df_selected[col] < mean - 3*std)]
    if len(outliers) > 0:
        print(f"   ⚠️ {col}: {len(outliers)} outliers")
        outlier_count += 1
if outlier_count == 0:
    print("   ✅ No se detectaron outliers")


🔍 Verificando rangos de valores:
   ✅ daily_screen_time_hours: [3.00, 12.00]
   ✅ sleep_hours: [4.50, 9.00]
   ✅ social_media_hours: [0.50, 6.00]
   ✅ gaming_hours: [0.00, 4.00]
   ✅ work_study_hours: [0.50, 6.00]
   ✅ notifications_per_day: [20.00, 250.00]
   ✅ app_opens_per_day: [15.00, 180.00]

🔍 Outliers detectados (>3 desviaciones):
   ⚠️ social_media_ratio: 110 outliers
   ⚠️ gaming_ratio: 111 outliers
   ⚠️ work_study_ratio: 108 outliers
   ⚠️ productivity_ratio: 141 outliers
   ⚠️ notification_intensity: 119 outliers
   ⚠️ sleep_screen_imbalance: 7 outliers
   ⚠️ entertainment_dominance: 186 outliers
   ⚠️ digital_compulsivity: 122 outliers


## 14. Guardado de Datos y Artefactos

In [20]:
# Guardar dataset limpio
dataset_limpio_path = os.path.join(PROCESADO_DIR, 'dataset_limpio.csv')
df_scaled.to_csv(dataset_limpio_path, index=False)
print(f"✅ Dataset limpio guardado en: {dataset_limpio_path}")
print(f"   Dimensiones: {df_scaled.shape}")

# Guardar scaler
scaler_path = os.path.join(ARTIFACTS_DIR, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ Scaler guardado en: {scaler_path}")

# Guardar label encoder
label_encoder_path = os.path.join(ARTIFACTS_DIR, 'label_encoder.pkl')
with open(label_encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"✅ Label Encoder guardado en: {label_encoder_path}")

# Guardar feature columns
feature_columns_path = os.path.join(ARTIFACTS_DIR, 'feature_columns.json')
with open(feature_columns_path, 'w') as f:
    json.dump({
        'feature_columns': FEATURE_COLUMNS,
        'target_column': 'addiction_level_encoded',
        'original_target': 'addiction_level',
        'class_mapping': {k: int(v) for k, v in class_mapping.items()},
        'raw_columns_for_risk_score': COLUMNAS_RAW
    }, f, indent=4)
print(f"✅ Feature columns guardado en: {feature_columns_path}")

# Guardar feature names para app.py
feature_names_path = os.path.join(ARTIFACTS_DIR, 'feature_names.pkl')
with open(feature_names_path, 'wb') as f:
    pickle.dump(FEATURE_COLUMNS, f)
print(f"✅ Feature names guardado en: {feature_names_path}")

✅ Dataset limpio guardado en: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/limpio_procesado/dataset_limpio.csv
   Dimensiones: (7500, 25)
✅ Scaler guardado en: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/artifacts/scaler.pkl
✅ Label Encoder guardado en: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/artifacts/label_encoder.pkl
✅ Feature columns guardado en: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/artifacts/feature_columns.json
✅ Feature names guardado en: /home/sarzuricarlos60/Desktop/Proyecto_Habitos_Digitales/datasets/artifacts/feature_names.pkl


## 15. Verificación Final

In [21]:
print("\n" + "="*60)
print("📊 VERIFICACIÓN FINAL")
print("="*60)

print(f"✅ Dataset shape: {df_scaled.shape}")
print(f"✅ Clases: {list(class_mapping.keys())}")
print(f"✅ Clases codificadas: {list(class_mapping.values())}")
print(f"✅ Features: {len(FEATURE_COLUMNS)}")
print(f"✅ Columnas _raw: {len(COLUMNAS_RAW)}")

# Verificaciones de integridad
assert df_scaled.isnull().sum().sum() == 0, "❌ Quedaron valores nulos"
assert df_scaled.duplicated().sum() == 0, "❌ Quedaron filas duplicadas"
assert len(class_mapping) == 4, "❌ Deberían existir 4 clases"
assert list(class_mapping.keys()) == ['None', 'Mild', 'Moderate', 'Severe'], "❌ Orden de severidad incorrecto"

print("\n✅ Todos los artefactos están listos para train.ipynb")
print("   - dataset_limpio.csv (datos normalizados)")
print("   - scaler.pkl (para normalizar nuevos datos)")
print("   - label_encoder.pkl (para codificar labels)")
print("   - feature_columns.json (metadatos del dataset)")
print("   - feature_names.pkl (nombres de features)")
print("   - dataset_statistics.json (estadísticas)")
print("   - Visualizaciones (correlación, distribuciones, boxplots)")


📊 VERIFICACIÓN FINAL
✅ Dataset shape: (7500, 25)
✅ Clases: ['None', 'Mild', 'Moderate', 'Severe']
✅ Clases codificadas: [0, 1, 2, 3]
✅ Features: 18
✅ Columnas _raw: 5

✅ Todos los artefactos están listos para train.ipynb
   - dataset_limpio.csv (datos normalizados)
   - scaler.pkl (para normalizar nuevos datos)
   - label_encoder.pkl (para codificar labels)
   - feature_columns.json (metadatos del dataset)
   - feature_names.pkl (nombres de features)
   - dataset_statistics.json (estadísticas)
   - Visualizaciones (correlación, distribuciones, boxplots)


## 16. Ejemplo de Uso de Artefactos

In [22]:
print("\n" + "="*60)
print("📝 EJEMPLO DE USO DE LOS ARTEFACTOS")
print("="*60)

with open(os.path.join(ARTIFACTS_DIR, 'feature_columns.json'), 'r') as f:
    feature_info = json.load(f)

print(f"📊 Features: {len(feature_info['feature_columns'])}")
print(f"🎯 Target: {feature_info['target_column']}")
print(f"📈 Clases: {feature_info['class_mapping']}")

print("\n✅ Todos los artefactos cargados correctamente")


📝 EJEMPLO DE USO DE LOS ARTEFACTOS
📊 Features: 18
🎯 Target: addiction_level_encoded
📈 Clases: {'None': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3}

✅ Todos los artefactos cargados correctamente
